In [15]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import monotonically_increasing_id
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, lit
from pyspark.sql.functions import (
    col,
    trim,
    lit,
    monotonically_increasing_id,
)
from functools import reduce

spark = SparkSession.builder.appName("normalize_to_silver").getOrCreate()

# <<< AQUÍ VA LA CONFIG DE BIGQUERY >>>
PROJECT_ID = "grupo2-essalud"
PLATA_DATASET = "essalud_plata"

# Bucket temporal que usará el conector para escribir en BigQuery
spark.conf.set("temporaryGcsBucket", "grupo2-essalud-datalake")

# ---------------------------
# CONFIG
# ---------------------------
silver_bucket = "gs://grupo2-essalud-datalake/plata"

# Tablas BRONCE en BigQuery
files = {
    "diabetes": "grupo2-essalud.essalud_bronce.Diabetes",
    "obesidad": "grupo2-essalud.essalud_bronce.Obesidad",
    "hipertension": "grupo2-essalud.essalud_bronce.Hipertension",
}

ubigeo_table = "grupo2-essalud.essalud_bronce.Ubigeo"

# Para guardar métricas de calidad
metricas = []   # cada elemento será un dict por tabla


# ---------------------------
# FUNCIÓN AUXILIAR: LIMPIEZA + MÉTRICAS
# ---------------------------
def limpiar_y_contar(df, nombre_tabla, columnas_clave):
    """
    - Cuenta total inicial
    - Cuenta registros con nulos en columnas_clave
    - Elimina nulos en columnas_clave
    - Elimina duplicados por columnas_clave
    - Devuelve df_limpio + métricas
    """

    # Total inicial
    total_inicial = df.count()

    # Nulos en columnas clave
    if columnas_clave:
        condicion_nulos = reduce(
            lambda acc, c: acc | col(c).isNull(),
            columnas_clave[1:],
            col(columnas_clave[0]).isNull()
        )
        nulos_eliminados = df.filter(condicion_nulos).count()
        df_sin_nulos = df.dropna(subset=columnas_clave)
    else:
        nulos_eliminados = 0
        df_sin_nulos = df

    # Duplicados
    if columnas_clave:
        antes_dups = df_sin_nulos.count()
        df_final = df_sin_nulos.dropDuplicates(columnas_clave)
        despues_dups = df_final.count()
        duplicados_eliminados = antes_dups - despues_dups
    else:
        df_final = df_sin_nulos
        duplicados_eliminados = 0

    # Guardar métricas en la lista global
    metricas.append({
        "tabla": nombre_tabla,
        "total_inicial": total_inicial,
        "nulos_eliminados": nulos_eliminados,
        "duplicados_eliminados": duplicados_eliminados,
        "total_final": df_final.count(),
    })

    print(
        f"[{nombre_tabla}] total_inicial={total_inicial}, "
        f"nulos_eliminados={nulos_eliminados}, "
        f"duplicados_eliminados={duplicados_eliminados}, "
        f"total_final={df_final.count()}"
    )

    return df_final


# ---------------------------
# LECTURA DE LAS 3 TABLAS BRONCE (BIGQUERY)
# ---------------------------
dfs = []
for enfermedad, table in files.items():
    df = (
        spark.read.format("bigquery")
        .option("table", table)
        .load()
    )

    # Normalizar strings (solo columnas string)
    tipos = dict(df.dtypes)
    for c in df.columns:
        if tipos[c] == "string":
            df = df.withColumn(c, trim(col(c)))

    # Añadir columna grupo_enfermedad según archivo
    if enfermedad == "diabetes":
        df = df.withColumn("grupo_enfermedad", lit("E1"))
    elif enfermedad == "obesidad":
        df = df.withColumn("grupo_enfermedad", lit("E2"))
    elif enfermedad == "hipertension":
        df = df.withColumn("grupo_enfermedad", lit("E3"))

    dfs.append(df)

df_all = (
    dfs[0]
    .unionByName(dfs[1], allowMissingColumns=True)
    .unionByName(dfs[2], allowMissingColumns=True)
)

# ---------------------------
# LECTURA UBIGEO DESDE BIGQUERY
# ---------------------------
ubigeo = (
    spark.read.format("bigquery")
    .option("table", ubigeo_table)
    .load()
)

ubigeo = ubigeo.select(
    col("Ubigeo").alias("ubigeo"),
    col("Departamento").alias("departamento"),
    col("Provincia").alias("provincia"),
    col("Distrito").alias("distrito"),
    col("Poblacion").alias("poblacion"),
)

# ---------------------------
# TABLA PACIENTE
# ---------------------------
paciente_raw = df_all.select(
    col("ID_PACIENTE").alias("id_paciente"),
    col("EDAD_PACIENTE").alias("edad_paciente"),
    col("SEXO_PACIENTE").alias("sexo_paciente"),
)

paciente = limpiar_y_contar(
    paciente_raw,
    "paciente",
    ["id_paciente"]
)

paciente.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv(f"{silver_bucket}/paciente")

# Tabla en BigQuery (Plata)
(
    paciente.write
    .format("bigquery")
    .option("table", f"{PROJECT_ID}.{PLATA_DATASET}.paciente")
    .mode("overwrite")
    .save()
)

# ---------------------------
# TABLA MEDICO
# ---------------------------
medico_raw = df_all.select(
    col("ID_MEDICO").alias("id_medico"),
    col("EDAD_MEDICO").alias("edad"),
)

medico = limpiar_y_contar(
    medico_raw,
    "medico",
    ["id_medico"]
)

medico.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv(f"{silver_bucket}/medico")

# Tabla en BigQuery (Plata)

(
    medico.write
    .format("bigquery")
    .option("table", f"{PROJECT_ID}.{PLATA_DATASET}.medico")
    .mode("overwrite")
    .save()
)

# ---------------------------
# TABLA CIE10
# ---------------------------
cie10_raw = df_all.select(
    col("COD_DIAG").alias("Cod_Enfermedad"),
    col("DIAGNOSTICO").alias("Des_Enfermedad"),
).dropDuplicates()

cie10 = limpiar_y_contar(
    cie10_raw,
    "CIE10",
    ["Cod_Enfermedad"]
)

cie10.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv(f"{silver_bucket}/cie10")

(
    cie10.write
    .format("bigquery")
    .option("table", f"{PROJECT_ID}:{PLATA_DATASET}.cie10")
    .mode("overwrite")
    .save()
)

# ---------------------------
# TABLA UBIGEO (PLATA)
# ---------------------------
ubigeo_plata_raw = ubigeo.select(
    col("ubigeo").alias("Ubigeo"),
    col("departamento").alias("Departamento"),
    col("provincia").alias("Provincia"),
    col("distrito").alias("Distrito"),
    col("poblacion").alias("poblacion"),
)

ubigeo_plata = limpiar_y_contar(
    ubigeo_plata_raw,
    "Ubigeo",
    ["Ubigeo"]
)

ubigeo_plata.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv(f"{silver_bucket}/ubigeo")

(
    ubigeo_plata.write
    .format("bigquery")
    .option("table", f"{PROJECT_ID}:{PLATA_DATASET}.ubigeo")
    .mode("overwrite")
    .save()
)

# ---------------------------
# TABLA DIAGNOSTICO (HECHOS)
# ---------------------------

diagnostico_raw = df_all.select(
    col("COD_DIAG").alias("Cod_Enfermedad"),
    col("ID_PACIENTE").alias("Id_paciente"),
    col("UBIGEO").alias("Ubigeo"),
    col("ID_MEDICO").alias("Id_Medico"),
    col("SERVICIO_HOSPITALARIO").alias("Servicio_Hospitalario"),
    col("ACTIVIDAD_HOSPITALARIA").alias("Actividad_Hospitalaria"),
    col("FECHA_MUESTRA").alias("Fecha_Muestra"),
)

diagnostico_clean = limpiar_y_contar(
    diagnostico_raw,
    "Diagnostico",
    ["Cod_Enfermedad", "Id_paciente", "Ubigeo"]
)


# Crear Cod_Diagnostico = 1,2,3,... (ID consecutivo)
w = Window.orderBy(lit(1))   # no importa el orden, solo queremos numerar

diagnostico_with_id = diagnostico_clean.withColumn(
    "Cod_Diagnostico",
    row_number().over(w)
)

# Reordenar columnas para que Cod_Diagnostico sea la primera
diagnostico = diagnostico_with_id.select(
    "Cod_Diagnostico",
    "Cod_Enfermedad",
    "Id_paciente",
    "Ubigeo",
    "Id_Medico",
    "Servicio_Hospitalario",
    "Actividad_Hospitalaria",
    "Fecha_Muestra",
)


diagnostico.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv(f"{silver_bucket}/diagnostico")


(
    diagnostico.write
    .format("bigquery")
    .option("table", f"{PROJECT_ID}:{PLATA_DATASET}.diagnostico")
    .mode("overwrite")
    .save()
)

# ---------------------------
# TABLA PROCEDIMIENTO
# ---------------------------
proc1 = df_all.select(
    col("PROCEDIMIENTO_1").alias("Des_Procedimiento")
).where(col("PROCEDIMIENTO_1").isNotNull())

proc2 = df_all.select(
    col("PROCEDIMIENTO_2").alias("Des_Procedimiento")
).where(col("PROCEDIMIENTO_2").isNotNull())

# Unimos y nos quedamos con valores únicos
procedimiento_desc = proc1.union(proc2).dropDuplicates()

# Creamos Cod_Procedimiento = 1,2,3,... (PK)
w_proc = Window.orderBy("Des_Procedimiento")

procedimiento = procedimiento_desc.withColumn(
    "Cod_Procedimiento",
    row_number().over(w_proc)
)

# Dejamos columnas en el orden del modelo
procedimiento = procedimiento.select(
    "Cod_Procedimiento",
    "Des_Procedimiento"
)

# Guardar en GCS (Plata)
procedimiento.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv(f"{silver_bucket}/procedimiento")

# Guardar en BigQuery (Plata)
(
    procedimiento.write
    .format("bigquery")
    .option("table", f"{PROJECT_ID}:{PLATA_DATASET}.procedimiento")
    .mode("overwrite")
    .save()
)

# ---------------------------
# TABLA RESULTADO_PROCEDIMIENTO (HECHOS)
# ---------------------------
# 1) Armamos los resultados incluyendo las llaves naturales del diagnóstico
res1 = df_all.select(
    col("COD_DIAG").alias("Cod_Enfermedad"),
    col("ID_PACIENTE").alias("Id_paciente"),
    col("UBIGEO").alias("Ubigeo"),
    col("FECHA_MUESTRA").alias("Fecha_Muestra"),
    col("PROCEDIMIENTO_1").alias("Des_Procedimiento"),
    col("RESULTADO_1").alias("Resultado"),
    col("UNIDADES_1").alias("Unidades"),
    col("FEC_RESULTADO_1").alias("Fecha_Resultado"),
)

res2 = df_all.select(
    col("COD_DIAG").alias("Cod_Enfermedad"),
    col("ID_PACIENTE").alias("Id_paciente"),
    col("UBIGEO").alias("Ubigeo"),
    col("FECHA_MUESTRA").alias("Fecha_Muestra"),
    col("PROCEDIMIENTO_2").alias("Des_Procedimiento"),
    col("RESULTADO_2").alias("Resultado"),
    col("UNIDADES_2").alias("Unidades"),
    col("FEC_RESULTADO_2").alias("Fecha_Resultado"),
)

resultado_raw = (
    res1.union(res2)
        .where(col("Des_Procedimiento").isNotNull())
)

# 2) JOIN con DIAGNOSTICO -> trae Cod_Diagnostico (PK)
resultado_join_diag = resultado_raw.join(
    diagnostico.select(
        "Cod_Diagnostico",
        "Cod_Enfermedad",
        "Id_paciente",
        "Ubigeo",
        "Fecha_Muestra",
    ),
    on=["Cod_Enfermedad", "Id_paciente", "Ubigeo", "Fecha_Muestra"],
    how="left"
)

# 3) JOIN con PROCEDIMIENTO -> trae Cod_Procedimiento (código numérico)
resultado_join_proc = resultado_join_diag.join(
    procedimiento.select(
        "Cod_Procedimiento",
        "Des_Procedimiento"
    ),
    on="Des_Procedimiento",
    how="left"
)

# 4) Nos quedamos SOLO con las columnas del modelo
resultado_sel = resultado_join_proc.select(
    "Cod_Procedimiento",   # ahora es el código numérico
    "Cod_Diagnostico",     # FK a Diagnostico
    "Resultado",
    "Unidades",
    "Fecha_Resultado",
)

# 5) Limpieza (nulos / duplicados)
resultado_clean = limpiar_y_contar(
    resultado_sel,
    "Resultado_Procedimiento",
    ["Cod_Procedimiento", "Cod_Diagnostico", "Fecha_Resultado"]
)

resultado_proc = resultado_clean

# 6) Guardar en GCS (Plata)
resultado_proc.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv(f"{silver_bucket}/resultado_procedimiento")

# 7) Guardar en BigQuery (Plata)
(
    resultado_proc.write
    .format("bigquery")
    .option("table", f"{PROJECT_ID}:{PLATA_DATASET}.resultado_procedimiento")
    .mode("overwrite")
    .save()
)


# ---------------------------
# GUARDAR MÉTRICAS DE CALIDAD EN CSV (PLATA)
# ---------------------------
metricas_df = spark.createDataFrame(metricas)

metricas_df.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv(f"{silver_bucket}/metricas_calidad")

spark.stop()


25/12/01 19:40:43 INFO SparkEnv: Registering MapOutputTracker
25/12/01 19:40:43 INFO SparkEnv: Registering BlockManagerMaster
25/12/01 19:40:43 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
25/12/01 19:40:43 INFO SparkEnv: Registering OutputCommitCoordinator


[paciente] total_inicial=1308325, nulos_eliminados=0, duplicados_eliminados=666000, total_final=642325


[medico] total_inicial=1308325, nulos_eliminados=0, duplicados_eliminados=1301635, total_final=6690


[CIE10] total_inicial=81, nulos_eliminados=0, duplicados_eliminados=0, total_final=81


[Ubigeo] total_inicial=1874, nulos_eliminados=0, duplicados_eliminados=0, total_final=1874


[Diagnostico] total_inicial=1308325, nulos_eliminados=0, duplicados_eliminados=397610, total_final=910715


25/12/01 19:43:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/01 19:43:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/01 19:43:47 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/01 19:43:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/01 19:43:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/01 19:44:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/01 1

[Resultado_Procedimiento] total_inicial=2616650, nulos_eliminados=794982, duplicados_eliminados=212, total_final=1821456


25/12/01 19:47:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/01 19:47:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/01 19:47:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/01 19:47:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/01 19:47:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/01 19:47:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/12/01 1

In [2]:
# ---------------------------
# TABLA DIAGNOSTICO (HECHOS)
# ---------------------------
from pyspark.sql.functions import monotonically_increasing_id

# ---------------------------
# TABLA DIAGNOSTICO (HECHOS)
# ---------------------------
diagnostico_raw = df_all.select(
    col("COD_DIAG").alias("Cod_Enfermedad"),        # ← Código CIE10
    col("ID_PACIENTE").alias("Id_paciente"),
    col("UBIGEO").alias("Ubigeo"),
    col("ID_MEDICO").alias("Id_Medico"),
    col("SERVICIO_HOSPITALARIO").alias("Servicio_Hospitalario"),
    col("ACTIVIDAD_HOSPITALARIA").alias("Actividad_Hospitalaria"),
    col("FECHA_MUESTRA").alias("Fecha_Muestra"),
)

# Limpiamos por las 3 claves principales de relación
diagnostico_clean = limpiar_y_contar(
    diagnostico_raw,
    "Diagnostico",
    ["Cod_Enfermedad", "Id_paciente", "Ubigeo"]
)

# AGREGAMOS ID PRIMARIA (clave surrogate)
diagnostico = diagnostico_clean.withColumn(
    "Diagnostico_ID",
    monotonically_increasing_id()
)

# Reordenamos columnas (opcional pero recomendado)
diagnostico = diagnostico.select(
    "Diagnostico_ID",
    "Cod_Enfermedad",
    "Id_paciente",
    "Ubigeo",
    "Id_Medico",
    "Servicio_Hospitalario",
    "Actividad_Hospitalaria",
    "Fecha_Muestra"
)

# Guardar en GCS como CSV
diagnostico.coalesce(1).write.mode("overwrite") \
    .option("header", "true") \
    .csv(f"{silver_bucket}/diagnostico")

# Guardar en BigQuery
(
    diagnostico.write
    .format("bigquery")
    .option("table", f"{PROJECT_ID}:{PLATA_DATASET}.diagnostico")
    .mode("overwrite")
    .save()
)


NameError: name 'df_all' is not defined